# Cheeky Rejection Sampling

In [1]:
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import pandas as pd

In [70]:
def makeSample():
    flu = stats.bernoulli.rvs(0.01)
    if flu == 1:
        Temp = stats.norm.rvs(101, 2)
    else:
        Temp = stats.norm.rvs(98, 2)

    return [flu, Temp]

In [71]:
def matches_pattern(sample, pattern, gt_flag = False):
    """
    Checks if a sample matches a pattern.
    pattern: list with values or None (None means 'any')
    For the temperature (assumed to be the second element), 
    if the difference is <= 0.1, it's considered a match.
    """
    flu, temp = sample
    flu_hat, temp_hat = pattern
    flu_match = (flu == flu_hat) if flu_hat is not None else True
    if temp_hat is None:
        temp_match = True
    else:
        if gt_flag:
            temp_match = temp >= temp_hat
        else:
            temp_match = abs(temp - temp_hat) <= 0.1
    return flu_match and temp_match

In [72]:
data = [makeSample() for _ in range(100000)]

In [75]:
# question a
# P(Temp = 103 | Flu) = P(Temp = 103, Flu) / P(Flu)
pattern_1 = [1, 103]
pattern_2 = [1, None]
count_pattern_1 = sum([1 for i in data if matches_pattern(i, pattern_1, gt_flag = True)])
count_pattern_2 = sum([1 for i in data if matches_pattern(i, pattern_2)])
print(count_pattern_1 / count_pattern_2)

0.15415821501014199


In [104]:
# question b
# P(flu = 1 | Temp = 100) = P(Temp = 100, Flu = 1) / P(Temp = 100)
pattern_1 = [1, 100]
pattern_2 = [None, 100]
count_pattern_1 = sum([1 for i in data if matches_pattern(i, pattern_1)])
count_pattern_2 = sum([1 for i in data if matches_pattern(i, pattern_2)])
print(count_pattern_1 / count_pattern_2)

0.011349306431273645


In [98]:
x = [i for i in data if matches_pattern(i, pattern_2)]

In [105]:
import numpy as np
from scipy.stats import norm

# Set random seed for reproducibility
np.random.seed(42)

# Number of samples
N = 10000
p_flu = 0.01

# Sample Flu status and temperature
Fs = np.random.rand(N) < p_flu
Ts = np.where(Fs,
              norm.rvs(loc=101, scale=2, size=N),
              norm.rvs(loc=98, scale=2, size=N))

# Cheeky Rejection Sampling: accept if |T - 100| <= 0.1
accepted = np.abs(Ts - 100) <= 0.1
num_accepted = np.sum(accepted)

# Estimate P(Flu=1 | |Temp - 100| <= 0.1)
posterior_cheeky = np.sum(Fs & accepted) / num_accepted

print(f"Accepted samples: {num_accepted} out of {N}")
print(f"Estimated P(Flu=1 | |Temp - 100| <= 0.1): {posterior_cheeky:.4f}")

Accepted samples: 253 out of 10000
Estimated P(Flu=1 | |Temp - 100| <= 0.1): 0.0040
